<a href="https://colab.research.google.com/github/dks1532/SNU_BigData_AI_Fintech/blob/main/LendingClub_%E1%84%89%E1%85%B5%E1%86%AF%E1%84%89%E1%85%B3%E1%86%B8_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LendingClub 데이터분석 전 과정 체험 — EDA부터 Sharpe Ratio 극대화까지

**오늘 세션의 목표는:**

> 데이터를 처음 받았을 때부터, "Sharpe ratio를 극대화하는 투자 전략"이 나오기까지의 **전체 데이터 분석 과정을 한 번 끝까지 경험**하는 것.

오늘 코드는 모두 완성되어 있습니다. 여러분이 할 일은 (1) 각 셀이 **무슨 일을 하는지 이해하고** (2) 실행해서 결과를 확인하고 (3) 중간중간 나오는 **[Question]** 질문을 생각해보는 것입니다.



**목차:**
| 단계 | 하는 일 | 질문 |
|---|---|---|
| 1. EDA | 데이터의 생김새 파악 | 이 데이터엔 뭐가 들어있나? |
| 2. 수익률 정의 | 도메인 지식을 숫자로 | 대출 투자의 '수익'이란 뭔가? |
| 3. Sharpe ratio | 수익과 위험을 하나의 지표로 | 어느 투자가 '위험 대비' 좋은가? |
| 4. 예측 모델 | 부도 확률 예측 (로지스틱 회귀) | 어떤 대출이 위험한가? |
| 5. 전략 최적화 | Sharpe를 극대화하는 기준 탐색 | 어디까지 승인해야 하나? |

---
# Part 0. 준비

**이 셀이 하는 일** : 오늘 쓸 도구를 불러옵니다.
- `pandas` : 표(데이터프레임)를 다루는 도구. 엑셀의 파이썬 버전이라고 생각하면 됩니다.
- `numpy` : 숫자 배열 계산 도구. `np.where`(조건 분기), `np.argmax`(최댓값 위치) 등을 씁니다.
- `matplotlib` : 그래프 그려주는 도구.
- `sklearn`에서는 로지스틱 회귀(`LogisticRegression`)와 데이터 분할기(`train_test_split`)만 빌려옵니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

**이 셀이 하는 일** : 데이터 파일을 준비합니다.

- **Colab**이라면: 아래 셀을 실행하면 **[파일 선택] 버튼**이 나타납니다. 버튼을 눌러 `lending_club.csv`를 선택하세요. (몇 초 이내에 업로드됩니다)

In [ ]:
import os

if not os.path.exists('lending_club.csv'):        # 파일이 아직 없으면
    from google.colab import files                 # Colab의 업로드 도구를 불러와서
    files.upload()                                 # [파일 선택] 버튼을 띄웁니다

**이 셀이 하는 일** : csv 파일을 읽어 `df`라는 데이터프레임(표)에 담습니다.

`lending_club.csv`는 미국 P2P 대출 플랫폼 LendingClub의 실제 데이터입니다. 원본은 컬럼이 145개인데, 오늘은 그 중 일부 11개만 정제한 버전을 씁니다. (여러분 프로젝트에서는 원본 데이터셋에서 어떤 컬럼을 어떻게 사용할지를 고민해야 합니다)

In [ ]:
df = pd.read_csv('lending_club.csv')

---
# Part 1. EDA — 데이터 탐색

새 데이터를 받으면 보통 다음 순서로 내용파악을 합니다: **크기 → 내용물 → 요약**

**이 셀이 하는 일** : `.shape`는 (행 수, 열 수), `.head()`는 위에서 5줄을 보여줍니다.

In [ ]:
print(df.shape)   # (행, 열) = (대출 건수, 변수 개수)
df.head()

각 컬럼의 의미:

| 컬럼 | 의미 |
|---|---|
| **`bad`** | **1 = 부도, 0 = 정상 상환. 오늘의 주인공(예측 목표)** |
| `loan_amnt` | 대출 금액 (\$) |
| `int_rate` | 이자율 (%) |
| `grade` | LendingClub이 매긴 신용등급 (A=우량 ~ G=위험) |
| `fico_range_low/high` | FICO 신용점수 (미국판 신용점수) |
| `dti` | 소득 대비 부채 비율 (Debt-To-Income) |
| `annual_inc` | 연소득 (\$) |
| `open_acc` / `total_acc` | 현재 열려있는 / 누적 신용계좌 수 |
| `inq_last_6mths` | 최근 6개월간 신용조회 횟수 (급전이 필요한 사람일수록 많겠죠?) |

**이 셀이 하는 일** : `.info()`는 각 컬럼의 타입과 결측치 유무를 한 번에 보여줍니다. `non-null` 개수가 행 수와 같으면 결측치가 없다는 뜻입니다. (오늘 데이터는 결측치가 없도록 정제해뒀지만, 원본을 다룰 땐 이 확인이 필수입니다 — `df.isna().sum()`도 같은 용도.)

In [ ]:
df.info()

**이 셀이 하는 일** : `value_counts()`는 값별 개수를 세줍니다. `normalize=True`를 주면 개수 대신 비율로 보여줍니다.

In [ ]:
df['bad'].value_counts(normalize=True)

약 **25%**, 네 건 중 한 건이 부도입니다. 예상보다 높습니다.

그런데도 사람들이 투자하는 이유 = **이자율**. 그럼 위험(등급)과 보상(이자율)이 어떻게 거래되고 있는지 봅시다.

**이 셀이 하는 일** : `groupby('grade')`는 데이터를 등급별 그룹으로 쪼개고, 그 뒤의 `.mean()`은 각 그룹 안에서 평균을 냅니다.

In [ ]:
avg_int = df.groupby('grade')['int_rate'].mean()      # 등급별 평균 이자율
default_rate = df.groupby('grade')['bad'].mean()      # 등급별 부도율 (0/1의 평균 = 비율!)

print(avg_int.round(2))
print()
print(default_rate.round(3))

**이 셀이 하는 일** : 두 결과를 나란히 막대그래프로 그립니다. `plot(kind='bar')`는 pandas에 내장된 간편 그래프 기능입니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
avg_int.plot(kind='bar', ax=axes[0], title='Average Interest Rate (%) by Grade')
default_rate.plot(kind='bar', ax=axes[1], color='tomato', title='Default Rate by Grade')
plt.tight_layout()
plt.show()

**여기까지의 발견** : 등급이 나빠질수록 이자율(보상)도, 부도율(위험)도 **함께** 올라갑니다. G등급은 이자율 31%지만 부도율이 55%.

그래서 질문이 생깁니다 — **"그래서 어디에 투자하는 게 최선인데?"**

이 질문에 답하려면 '수익'과 '위험'을 **하나의 숫자**로 합쳐야 합니다. 다음 파트에서 그 숫자(Sharpe ratio)를 직접 만들어봅니다.

---
# Part 2. 수익률 정의

모델링보다 먼저 해야 하는 것: **"수익률이 뭔지"를 직접 정의**하는 일입니다. 오늘은 단순화 버전으로:

- 정상 상환(`bad == 0`) → 이자율만큼 벌었다 : `int_rate / 100` (%를 소수로)
- 부도(`bad == 1`) → 수익 0 (원금은 회수됐다고 단순화)

**이 셀이 하는 일** : `np.where(조건, A, B)`는 행마다 "조건이 참이면 A, 거짓이면 B"를 넣습니다. 엑셀의 IF 함수와 똑같습니다. 이걸로 `realized`(실현 수익률)라는 새 컬럼을 만듭니다.

> 이 수익률 정의는 프로젝트에서 여러분이 **직접 정의해야 하는 부분**입니다. 부도나면 정말 0일까요? 원금 일부만 잃지 않을까요? (`total_pymnt` 같은 원본 컬럼을 쓰면 실제 회수액 기반으로 계산할 수 있습니다.)

In [ ]:
df['realized'] = np.where(df['bad'] == 0, df['int_rate'] / 100, 0.0)

# 잘 만들어졌는지 눈으로 확인 — bad=0이면 이자율이, bad=1이면 0이 들어가야 함
df[['grade', 'int_rate', 'bad', 'realized']].head()

---
# Part 3. Sharpe Ratio

$$\text{Sharpe ratio} = \frac{E[R] - R_f}{\sigma} \qquad \begin{cases} E[R] = \text{평균 수익률} \\ R_f = \text{무위험수익률 (5% 가정)} \\ \sigma = \text{수익률의 표준편차 (위험)} \end{cases}$$

읽는 법: **"위험 1단위를 감수할 때마다 무위험 대비 초과수익을 얼마나 받는가."** 분자가 커도 분모(변동성)가 크면 좋은 투자가 아닙니다.

**이 셀이 하는 일** : `agg(['mean','std'])`는 등급별로 평균과 표준편차를 **한 번에** 계산합니다. 그리고 Sharpe 공식을 그대로 한 줄로 옮깁니다.

**[Question 1]** **Sharpe ratio 1등은 어느 등급일까요?** (평균 수익률 1등은 F/G 근처일 겁니다. Sharpe도 그럴까요?)

In [ ]:
rf = 0.05   # 무위험수익률 5% 가정

stats = df.groupby('grade')['realized'].agg(['mean', 'std'])
stats['sharpe'] = (stats['mean'] - rf) / stats['std']
stats.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
stats['mean'].plot(kind='bar', ax=axes[0], title='Mean Realized Return by Grade')
stats['sharpe'].plot(kind='bar', ax=axes[1], color='seagreen', title='Sharpe Ratio by Grade')
plt.tight_layout()
plt.show()

## 첫 번째 관찰

평균 수익률 1등은 F(약 14%)입니다. 그런데 **Sharpe ratio 1등은 B등급**입니다.

F, G등급은 부도가 절반이라 수익이 '이자 30% 아니면 0'으로 널뛰기합니다 — 표준편차(분모)가 너무 커서, 위험 대비로 보면 오히려 나쁜 투자인 것입니다. 이것이 프로젝트가 '평균수익'이 아니라 'Sharpe'를 목적함수로 삼는 이유입니다.

**[Question 2]** 위 셀에서 `rf = 0.05`를 `0.03`이나 `0.08`로 바꿔 재실행해보세요. 무위험수익률이 높아지면 어떤 등급이 유리/불리해지는지 파악해보시기 바랍니다. (안전자산의 매력이 커지면, 위험을 감수할 가치가 있는 등급이 줄어듭니다)

---

그런데 "B등급에 몰빵"은 아직 거친 전략입니다. 같은 B등급 안에도 갚을 사람과 못 갚을 사람이 섞여 있으니까요. **대출 1건 1건의 부도 확률을 예측해서 위험한 것만 골라낼 수 있다면?** — 예측 모델을 만들어봅시다

---
# Part 4. 부도 예측 모델 — 로지스틱 회귀


우리가 예측하려는 것은 **"부도가 날 확률"**, 즉 0과 1 사이의 숫자입니다. 이런 문제에 쓰는 표준 도구가 **로지스틱 회귀**입니다:

$$P(\text{bad}=1 \mid X) \;=\; \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_k x_k)}}$$

생김새가 복잡해 보이지만 구조는 단순합니다. 괄호 안($\beta_0 + \beta_1 x_1 + \cdots$)은 우리가 아는 **선형회귀식 그대로**입니다. 다만 선형회귀 값은 $-\infty \sim +\infty$ 어디든 갈 수 있어서 확률로 쓰기 어렵습니다. 그래서 그 값을 **시그모이드 함수** $\frac{1}{1+e^{-z}}$에 통과시켜 **0과 1 사이로 구겨 넣는 것**입니다.

- 선형식 값이 아주 크면 → $e^{-z} \to 0$ → 확률 $\to 1$ (부도 확실)
- 선형식 값이 아주 작으면(음수) → $e^{-z} \to \infty$ → 확률 $\to 0$ (정상 확실)
- 선형식 값이 0이면 → 확률 정확히 0.5 (반반)


**모델 학습 방법**은 다음과 같습니다:
```
전체 데이터 ──▶ Train(80%) ──▶ 진짜 Train(75%) : 모델 학습
          │              └──▶ Validation(25%) : threshold 결정
          └──▶ Test(20%) : 마지막에 딱 한 번, 성적표 확인 (out-of-sample test)
```

**왜 세 조각으로 나누나요?** Test를 미리 훔쳐보며 전략을 고르면, 그건 답안지를 보고 시험을 친 것과 같습니다. Test는 절대 모델 선택에 사용해서는 안되고(out of sample test), 모델 학습 전략 선택은 전부 Validation에서 끝냅니다.

**이 셀이 하는 일** : 모델은 문자를 이해하지 못하기 때문에, 범주형 변수 `grade`(A~G)를 0/1 더미변수들로 바꿉니다(`get_dummies`). `drop_first=True`로 첫 범주(A)를 기준으로 삼아 떨어뜨립니다. `realized`는 답을 미리 알려주는 변수라 반드시 제외합니다.

In [ ]:
df_model = pd.get_dummies(df.drop(columns=['realized']), columns=['grade'], drop_first=True)
df_model.head(3)   # grade 컬럼이 grade_B ~ grade_G 더미들로 바뀐 것을 확인

**이 셀이 하는 일** : `train_test_split`으로 두 번 자릅니다. `random_state=111`은 "자르는 방식을 고정"하는 옵션 — 누가 실행해도 같은 결과가 나오게 합니다(재현성). X는 재료(설명변수들), y는 정답(`bad`)입니다.

In [ ]:
# 1차 분할: 전체 → Train 80% / Test 20%
train, test = train_test_split(df_model, test_size=0.2, random_state=111)

X_test = test.drop('bad', axis=1);  y_test = test['bad']
X      = train.drop('bad', axis=1); y      = train['bad']

# 2차 분할: Train → 진짜 Train 75% / Validation 25%
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=111)

print(f"train {len(X_train)}건 / validation {len(X_val)}건 / test {len(X_test)}건")

**이 셀이 하는 일** : 로지스틱 회귀를 학습시키고 확률을 예측합니다. sklearn의 모든 모델은 같은 인터페이스를 씁니다:

- `.fit(X, y)` : "이 재료(X)와 정답(y)으로 패턴을 배워라" — **학습**
- `.predict_proba(X)` : "새 데이터의 **확률**을 내놓아라" — 결과는 [정상일 확률, 부도일 확률] 두 열이므로, `[:, 1]`로 **부도 확률 열만** 꺼냅니다.

> `solver='newton-cholesky'`는 계수를 찾는 계산 방법의 지정입니다. 연소득처럼 단위가 큰 변수가 섞여 있으면 기본 계산법은 수렴 경고를 내는데, 이 방법은 우리 데이터 크기에서 빠르고 조용하게 수렴합니다. (모델 자체가 달라지는 건 아닙니다)

In [ ]:
logit = LogisticRegression(solver='newton-cholesky')
logit.fit(X_train, y_train)                       # 학습: 6,000건으로 모델 학습

X_val_pred = logit.predict_proba(X_val)[:, 1]     # 예측: validation 2,000건의 '부도 확률'
X_val_pred[:10].round(3)

예측 확률이 나왔습니다. 하지만 확률만으로는 투자를 못 합니다 — **결정**이 필요합니다.

> "예측 부도확률이 ___ 이상이면 거절한다"

이 빈칸의 숫자가 **threshold(임계값)** 입니다. 낮추면 깐깐한 투자자(많이 거절), 높이면 공격적 투자자(많이 승인). **그럼 최적의 threshold는?** — "Sharpe ratio가 최대가 되는 지점"입니다.

---
# Part 5. Sharpe를 극대화하는 threshold 찾기

투자 전략의 수익 구조를 표로 정리하면:

| 내 결정 | 실제 결과 | 내 수익률 |
|---|---|---|
| 승인 | 정상 상환 | `int_rate` |
| 승인 | 부도 | 0 |
| 거절 | (상관없음) | 무위험 5% (그 돈은 국채에) |

**이 셀이 하는 일** : 위 표를 코드로 그대로 옮긴 `sharpe_ratio()` 함수를 정의합니다. `np.where`가 두 번 나옵니다 — 한 번은 "실제 결과"의 분기, 한 번은 "내 결정"의 분기. Part 2에서 배운 바로 그 함수입니다.

> 참고: 예전 실습 자료에는 같은 함수가 `sharp`라는 이름으로 있습니다. 내용은 동일합니다.

In [ ]:
def sharpe_ratio(y_true, y_pred, X, risk_free=0.05):
    """투자 전략(y_pred: 0=승인, 1=거절)의 Sharpe ratio를 계산"""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred).astype(int)
    int_rate = np.asarray(X['int_rate']) / 100

    loan_return     = np.where(y_true == 0, int_rate, 0.0)         # 실제 결과: 정상이면 이자, 부도면 0
    strategy_return = np.where(y_pred == 0, loan_return, risk_free) # 내 결정: 승인이면 위 값, 거절이면 5%

    excess = strategy_return - risk_free    # 무위험 대비 초과수익
    std = excess.std(ddof=1)                # ddof=1: 표본표준편차
    if std == 0:
        return 0.0
    return excess.mean() / std

**이 셀이 하는 일** : threshold를 0.01부터 0.99까지 99개 지점으로 움직이며, 각 지점에서의 **validation** Sharpe를 계산합니다. for 반복문이 하는 일은 단순합니다 — "기준을 t로 잡아보고 → 그 전략의 Sharpe 재보고 → 기록" × 99번.

`(X_val_pred > t)`는 True/False 배열이 되고, `.astype(int)`로 1/0(거절/승인)이 됩니다.

In [ ]:
thresholds = np.linspace(0.01, 0.99, 99)
sharpe_scores = []

for t in thresholds:
    decision = (X_val_pred > t).astype(int)                    # 부도확률이 t 초과 → 거절(1)
    sharpe_scores.append(sharpe_ratio(y_val, decision, X_val))

sharpe_scores = np.array(sharpe_scores)

**이 셀이 하는 일** : 99개 점을 이어 **Sharpe 곡선**을 그리고, `np.argmax`(최댓값의 *위치*를 알려주는 함수)로 봉우리를 찾습니다.

**[Question 3]** 실행 전에 — threshold가 아주 작을 때(전부 거절)와 아주 클 때(전부 승인), Sharpe는 각각 어떻게 될까요?

In [ ]:
best_idx = np.argmax(sharpe_scores)
optimal_threshold = thresholds[best_idx]

plt.figure(figsize=(8, 4))
plt.plot(thresholds, sharpe_scores)
plt.axvline(optimal_threshold, color='red', linestyle='--',
            label=f'optimal t = {optimal_threshold:.2f}')
plt.xlabel('Threshold'); plt.ylabel('Sharpe ratio (validation)')
plt.title('Sharpe Ratio vs. Threshold')
plt.legend(); plt.show()

print(f"최적 threshold : {optimal_threshold:.2f}")
print(f"그때의 validation Sharpe : {sharpe_scores[best_idx]:.4f}")

**이 셀이 하는 일** : 마지막 관문, **out-of-sample test**. 금고에 넣어뒀던 Test set을 꺼내 딱 한 번 성적을 확인합니다.

순서에 주의: ① Train **전체**(X, y)로 모델을 다시 학습(데이터를 최대한 활용) → ② validation에서 정한 threshold를 **그대로** 적용 → ③ '아무 생각 없이 전부 승인' 전략과 비교.

In [ ]:
logit_final = LogisticRegression(solver='newton-cholesky').fit(X, y)   # ① Train 전체로 재학습
X_test_pred = logit_final.predict_proba(X_test)[:, 1]

decision_test = (X_test_pred > optimal_threshold).astype(int)          # ② 정해둔 기준 적용
model_sharpe = sharpe_ratio(y_test, decision_test, X_test)

all_approve = np.zeros(len(y_test))                                    # ③ 비교 기준: 전부 승인
naive_sharpe = sharpe_ratio(y_test, all_approve, X_test)

print(f"모델 전략의  Test Sharpe : {model_sharpe:.4f}")
print(f"전부 승인의 Test Sharpe : {naive_sharpe:.4f}")

## 두 번째 (그리고 더 중요한) 관찰

Validation에서는 모델이 분명 더 좋아 보였는데(약 0.68), **Test에서는 '전부 승인'과 사실상 동률**입니다. 두 가지 관찰을 한다면:

**1. 이것이 out-of-sample test의 존재 이유입니다.** Validation 성능만 믿는다면 "우리 모델이 좋다"고 믿게 됩니다. 그러나 진짜 성능은 한 번도 보지 않은 out of sample test 데이터에서만 확인됩니다 — 프로젝트에서도 test 성능으로 성능을 보고해야 합니다.

**2. 이것이 이 프로젝트의 출발선입니다.** 지금 모델 구성을 보면: 변수 11개(원본은 145개), 기본 로지스틱 회귀, 가장 단순한 수익률 정의. 이 세 가지가 대표적으로 개선할 수 있는 부분입니다(이외에도 많이 있습니다)

---
# 표본분할을 바꿔서 반복해보기

방금 확인한 Test Sharpe는 **random_state=111이라는 한 번의 분할**에서 나온 숫자입니다. 만약 데이터를 다르게 잘랐다면? 운 좋게 맞추기 쉬운 대출들이 test로 갔을 수도, 그 반대일 수도 있습니다. 즉 이 숫자는 **추첨 한 번의 결과**입니다.

**이 셀이 하는 일** : 분할 시드를 0~29로 바꿔가며 **전체 파이프라인**(분할 → 학습 → validation에서 threshold 선택 → 재학습 → test 평가)을 30번 반복하고, test 에서 '전부 승인' baseline과의 sharpe ratio **차이**를 기록합니다.

In [ ]:
logit_sharpes = []
baseline_sharpes = []

for seed in range(30):                          # 시드 0~29로 30번 반복
    # 1. 분할 (매번 다르게)
    tr, te = train_test_split(df_model, test_size=0.2, random_state=seed)
    X_te, y_te = te.drop('bad', axis=1), te['bad']
    X_full, y_full = tr.drop('bad', axis=1), tr['bad']
    X_tr, X_v, y_tr, y_v = train_test_split(X_full, y_full, test_size=0.25, random_state=seed)

    # 2. 학습 → validation에서 threshold 선택
    m = LogisticRegression(solver='newton-cholesky').fit(X_tr, y_tr)
    p_v = m.predict_proba(X_v)[:, 1]
    scores = [sharpe_ratio(y_v, (p_v > t).astype(int), X_v) for t in thresholds]
    t_opt = thresholds[np.argmax(scores)]

    # 3. Train 전체로 재학습 → test 평가
    m_final = LogisticRegression(solver='newton-cholesky').fit(X_full, y_full)
    p_te = m_final.predict_proba(X_te)[:, 1]
    logit_sharpes.append(sharpe_ratio(y_te, (p_te > t_opt).astype(int), X_te))
    baseline_sharpes.append(sharpe_ratio(y_te, np.zeros(len(y_te)), X_te))   # 같은 test에서의 baseline(전부 승인) 샤프비율 성능

logit_sharpes = np.array(logit_sharpes)
baseline_sharpes = np.array(baseline_sharpes)
diff = logit_sharpes - baseline_sharpes         # 같은 분할끼리 차이 계산

print(f"Logistic Regression Sharpe : 평균 {logit_sharpes.mean():.4f} (표준편차 {logit_sharpes.std():.4f})")
print(f"Baseline Sharpe (전부 승인) : 평균 {baseline_sharpes.mean():.4f} (표준편차 {baseline_sharpes.std():.4f})")
print(f"차이 (logit − baseline)     : 평균 {diff.mean():+.4f}, 범위 {diff.min():+.4f} ~ {diff.max():+.4f}")
print(f"Logistic Regression이 이긴 비율: {(diff > 0).mean():.0%}")